In [ ]:
# TakeMeter — r/smashbros Data Collection Script
# Run this in Google Colab (Runtime > Run all)
# Output: smashbros_unlabeled.csv ready for annotation

# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install datasets pandas -q

# ── Step 2: Imports ────────────────────────────────────────────────────────────
import pandas as pd
from datasets import load_dataset

# ── Step 3: Load the dataset ───────────────────────────────────────────────────
# Using "sentence-transformers/reddit-title-body" — a clean, modern Reddit
# dataset in Parquet format (no loading script, no trust_remote_code needed).
# Contains post titles + bodies with subreddit labels.

print("Loading dataset... (this may take 1-2 minutes)")

dataset = load_dataset(
    "sentence-transformers/reddit-title-body",
    split="train",
    streaming=True   # stream so we don't download the entire ~20GB dataset
)

print("Scanning for r/smashbros posts (streaming, may take 2-4 minutes)...")

# ── Step 4: Filter to r/smashbros ─────────────────────────────────────────────
smash_rows = []
checked = 0

for row in dataset:
    checked += 1
    if row.get("subreddit", "").lower() == "smashbros":
        smash_rows.append(row)

    # Progress update every 100k rows
    if checked % 100_000 == 0:
        print(f"  Checked {checked:,} posts... found {len(smash_rows)} smashbros so far")

    # Stop once we have enough, or after scanning 5M rows
    if len(smash_rows) >= 500 or checked >= 5_000_000:
        break

print(f"\nDone scanning. Found {len(smash_rows)} r/smashbros posts.")

# ── Step 5: Convert to DataFrame and clean ────────────────────────────────────
df = pd.DataFrame(smash_rows)
print(f"Columns available: {list(df.columns)}")

# Combine title + body into one text field
def combine_text(row):
    title = str(row.get("title", "") or "").strip()
    body  = str(row.get("body", "") or "").strip()
    if len(body) < 20 or body.lower() == title.lower():
        return title
    return f"{title}\n\n{body}"

df["text"] = df.apply(combine_text, axis=1)

# Drop posts where combined text is too short
df = df[df["text"].str.len() >= 30].copy()
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Posts after cleaning: {len(df)}")

# ── Step 6: Sample and prepare for annotation ─────────────────────────────────
if len(df) > 400:
    df_sample = df.sample(n=400, random_state=42).reset_index(drop=True)
    print("Sampled down to 400 posts.")
else:
    df_sample = df.copy()
    print(f"Using all {len(df_sample)} posts found.")

df_sample["label"] = ""
df_sample["notes"] = ""
df_final = df_sample[["text", "label", "notes"]].copy()

# ── Step 7: Save to CSV ────────────────────────────────────────────────────────
output_path = "smashbros_unlabeled.csv"
df_final.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")
print(f"Total rows to annotate: {len(df_final)}")
print("\nLabels to use: analysis | opinion | help")
print("Delete rows that are purely video/image posts with no real text.")

# ── Preview first 5 posts ──────────────────────────────────────────────────────
print("\n── Sample posts ──────────────────────────────────────────────────")
for i, row in df_final.head(5).iterrows():
    preview = row["text"][:250]
    print(f"\n[{i+1}] {preview}{'...' if len(row['text']) > 250 else ''}")
    print("     label: ___________")

Loading dataset... (this may take 1-2 minutes)


README.md:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Scanning for r/smashbros posts (streaming, may take 2-4 minutes)...
  Checked 100,000 posts... found 0 smashbros so far
  Checked 200,000 posts... found 0 smashbros so far
  Checked 300,000 posts... found 0 smashbros so far
  Checked 400,000 posts... found 0 smashbros so far
  Checked 500,000 posts... found 0 smashbros so far
  Checked 600,000 posts... found 0 smashbros so far
  Checked 700,000 posts... found 4 smashbros so far
  Checked 800,000 posts... found 10 smashbros so far
  Checked 900,000 posts... found 11 smashbros so far
  Checked 1,000,000 posts... found 15 smashbros so far
  Checked 1,100,000 posts... found 20 smashbros so far
  Checked 1,200,000 posts... found 26 smashbros so far
  Checked 1,300,000 posts... found 29 smashbros so far
  Checked 1,400,000 posts... found 36 smashbros so far
  Checked 1,500,000 posts... found 38 smashbros so far
  Checked 1,600,000 posts... found 40 smashbros so far
  Checked 1,700,000 posts... found 44 smashbros so far
  Checked 1,800,000 po

In [5]:
# TakeMeter — Groq Pre-labeling Script (with resume support)
# Run this in Google Colab
# - First run: labels all 400 rows (may hit daily token limit partway through)
# - Resume run: upload the partially-labeled CSV, it skips already-labeled rows

!pip install groq pandas -q

import pandas as pd
import time
from groq import Groq
from google.colab import files, userdata

# ── API key ────────────────────────────────────────────────────────────────────
try:
    api_key = userdata.get("GROQ_API_KEY")
except:
    api_key = input("Paste your Groq API key: ").strip()

client = Groq(api_key=api_key)

# ── Upload CSV ─────────────────────────────────────────────────────────────────
print("Upload your CSV (smashbros_unlabeled.csv OR a partially-labeled file)...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"Loaded {len(df)} rows.")

# ── Count already-labeled rows ─────────────────────────────────────────────────
already_done = df["label"].notna() & (df["label"] != "") & (df["label"] != "error")
print(f"Already labeled: {already_done.sum()} rows")
print(f"Rows to label:   {(~already_done).sum()} rows")

# ── Label definitions ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a classifier for posts from the r/smashbros subreddit.
Classify each post into EXACTLY ONE of these three labels:

analysis - Makes a structured argument backed by specific mechanical, statistical,
or competitive evidence. The claim is supported by verifiable details such as frame
data, matchup numbers, tournament results, or named mechanics — not just asserted.
Examples: matchup charts, tier list breakdowns with reasoning, frame data discussions,
character comparisons with specific evidence.

opinion - States a bold or evaluative position WITHOUT supporting evidence. The post
asserts rather than argues. The claim might be correct, but no real reasoning is given.
Examples: "X character is broken", "Melee is better than Brawl", roster wishlists,
feature requests with no supporting reasoning.

help - Asks for or offers practical gameplay advice with no evaluative claim about the
game. The goal is to solve a specific problem or improve at a specific skill.
Examples: "how do I wavedash", "tips for playing Fox", "why does my Project M freeze",
hardware questions, setup questions, finding players.

If the post is primarily a tournament announcement, player finder, community survey,
or has no real text argument, output: delete

Output ONLY one word: analysis, opinion, help, or delete
No explanation, no punctuation, just the single label word."""

def classify_post(text):
    truncated = text[:1500] if len(text) > 1500 else text
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this post:\n\n{truncated}"}
            ],
            max_tokens=10,
            temperature=0
        )
        label = response.choices[0].message.content.strip().lower()
        valid = {"analysis", "opinion", "help", "delete"}
        if label not in valid:
            for v in valid:
                if v in label:
                    return v
            return "review"
        return label
    except Exception as e:
        print(f"  API error: {e}")
        return "error"

# ── Classify only unlabeled rows ───────────────────────────────────────────────
to_label = df[~already_done].index.tolist()
print(f"\nStarting classification of {len(to_label)} rows...")

labeled_count = 0
for i, idx in enumerate(to_label):
    label = classify_post(str(df.at[idx, "text"]))
    df.at[idx, "label"] = label
    df.at[idx, "pre_labeled"] = "groq"
    labeled_count += 1

    if labeled_count % 25 == 0:
        counts = df["label"].value_counts().dropna().to_dict()
        print(f"  [{labeled_count}/{len(to_label)}] {counts}")
        # Save progress checkpoint in case of another rate limit hit
        df.to_csv("smashbros_prelabeled.csv", index=False)

    time.sleep(2)

# ── Final save ─────────────────────────────────────────────────────────────────
df.to_csv("smashbros_prelabeled.csv", index=False)

print("\n── Final label distribution ──────────────────────────────────────────")
print(df["label"].value_counts())

usable = df[~df["label"].isin(["delete", "review", "error", None])]
print(f"\nUsable posts (non-delete): {len(usable)}")
for label in ["analysis", "opinion", "help"]:
    count = (usable["label"] == label).sum()
    if count < 60:
        print(f"⚠️  '{label}' only has {count} posts — you may need more")

print("\n✓ Saved to smashbros_prelabeled.csv")
print("Download from the Files panel, review every row, fix wrong labels.")

Upload your CSV (smashbros_unlabeled.csv OR a partially-labeled file)...


Saving smashbros_unlabeled.csv to smashbros_unlabeled (3).csv
Loaded 400 rows.
Already labeled: 0 rows
Rows to label:   400 rows

Starting classification of 400 rows...


/tmp/ipykernel_670/1370770528.py:90: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'delete' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "label"] = label


  [25/400] {'help': 17, 'opinion': 7, 'delete': 1}
  [50/400] {'help': 34, 'opinion': 11, 'delete': 5}
  [75/400] {'help': 50, 'opinion': 17, 'delete': 7, 'analysis': 1}
  [100/400] {'help': 63, 'opinion': 24, 'delete': 9, 'analysis': 4}
  [125/400] {'help': 82, 'opinion': 28, 'delete': 10, 'analysis': 5}
  [150/400] {'help': 98, 'opinion': 36, 'delete': 11, 'analysis': 5}
  [175/400] {'help': 114, 'opinion': 41, 'delete': 14, 'analysis': 6}
  [200/400] {'help': 127, 'opinion': 50, 'delete': 17, 'analysis': 6}
  [225/400] {'help': 139, 'opinion': 58, 'delete': 22, 'analysis': 6}
  [250/400] {'help': 154, 'opinion': 65, 'delete': 25, 'analysis': 6}
  [275/400] {'help': 167, 'opinion': 72, 'delete': 29, 'analysis': 7}
  [300/400] {'help': 184, 'opinion': 76, 'delete': 33, 'analysis': 7}
  [325/400] {'help': 199, 'opinion': 83, 'delete': 36, 'analysis': 7}
  [350/400] {'help': 213, 'opinion': 90, 'delete': 40, 'analysis': 7}
  [375/400] {'help': 229, 'opinion': 94, 'delete': 45, 'analysis